In [0]:
%run ./main

In [0]:
# Explore the data

orders_df.printSchema()
orders_df.show()

# total order count
print("Total orders:", orders_df.count())

In [0]:
customer_dataframe.show()

In [0]:
# Select + Filter + add colums
from pyspark.sql import functions as F
customer_dataframe.withColumn("signup_year", F.year(F.to_date("signup_date"))) \
    .select("name", "country", "signup_year").filter(F.col("country") == "US") \
    .show()

In [0]:
# Aggregation (Total Revenue per Product)

orders_enriched = orders_df.join(products_dataframe, "product_id", "left")
orders_enriched.show()


revenue_by_product = orders_enriched.withColumn(
    "revenue", F.col("quantity") * F.col("price")
).groupBy("product_id", "title").agg(F.sum("revenue").alias("total_revenue"))

revenue_by_product.show()



In [0]:
# Joins (with Broadcast Hint)
from pyspark.sql.functions import broadcast

orders_with_customer = orders_df.join(broadcast(customer_dataframe), orders_df.customer_id == customer_dataframe.customer_id, "left")

orders_with_customer.select("order_id", "name", "country", "status").show()

In [0]:
# Window Functions (Ranking)
from pyspark.sql.window import Window

w = Window.partitionBy("customer_id").orderBy(F.to_date("order_date").desc())
orders_df.withColumn("rank", F.row_number().over(w)).show()

In [0]:
# UDF vs Native Functions
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

@udf(returnType=StringType())
def product_tier(price):
    return "Premium" if price > 50 else "Regular"

# Using UDF
products_dataframe.withColumn("tier_udf", product_tier("price")).show()

# Native
products_dataframe.withColumn(
    "tier_native",
    F.when(F.col("price") > 50, "Premium").otherwise("Regular")
).show()

In [0]:
# Cache DataFrames

# orders_enriched.cache()
# Note: .cache() is not supported on serverless compute
orders_enriched.count()

In [0]:
# Write to Parquet (Partitioned)

orders_df.write.mode("overwrite").partitionBy("order_date").parquet(
    "/Volumes/pyspark_cat/ecommerce/ecom_volume"
)

In [0]:
# Total revenue generated by a customer - life time value
ltv = orders_enriched.withColumn(
        "revenue", 
        (F.col("quantity") * F.col("price")).cast("decimal(12, 2)")
    ).groupBy("customer_id").agg(
        F.collect_list("revenue").alias("revenues"),
        F.sum("revenue").alias("total_revenue"),
        F.max("order_date").alias("last_order_date")
)

ltv.show()